# 10 — The Bermudan's exposure, CVA sensitivities, and wrong-way risk

Three things that go beyond "simulate and average":

1. the exposure of a product whose future value depends on a decision not yet made;
2. what a CVA desk hedges with, and what it costs to compute;
3. what happens when default and exposure are not independent.

This notebook reads the same tables as notebook 09 (cached if that one has been run).

In [ ]:
import _util
from _util import run_analysis
import numpy as np, pandas as pd, matplotlib.pyplot as plt

t = run_analysis("xva")

## 1. An option that turns into a swap

A swap's future value is a known function of the model's state. A Bermudan's is not: on each path, at each date, the
holder either still has the option (worth the larger of exercising and continuing, never negative) or has already
exercised and owns the remaining swap (which can go negative).

The continuation value is computed as a function of the state on every exercise date by backward induction, and the
paths are then exercised optimally. The same simulation must therefore reproduce the price.

In [ ]:
b, s = t["xva_bermudan"], t["xva_bermudan_summary"].iloc[0]
print(f"price: grid {s.grid_price_bp:.2f} bp, paths {s.simulated_price_bp:.2f} ± {s.simulated_se_bp:.2f} bp, PDE {s.pde_price_bp:.2f} bp")
plt.stackplot(b.time, b.option_epe_bp, b.swap_epe_bp, labels=["still an option", "exercised: the remaining swap"], alpha=0.85)
plt.plot(b.time[1:], b.underlying_swap_epe_bp[1:], "o--", color="k", lw=1, label="the underlying swap held from today")
plt.xlabel("years"); plt.ylabel("discounted EPE, bp of notional"); plt.legend(); plt.show()
print(f"CVA: Bermudan {s.cva_bp:.1f} bp, swap {s.swap_cva_bp:.1f} bp      DVA: Bermudan {s.dva_bp:.1f} bp, swap {s.swap_dva_bp:.1f} bp")

**Hand check: a martingale.** Before any coupon has been paid, the deflated value of the position is a martingale,
so its mean on the first exercise date must be today's price. On that date nothing has been exercised earlier, so
the position is worth $\max(\text{exercise}, \text{continuation}) \ge 0$ on every path and EPE is that mean.

In [ ]:
print(f"EPE on the first exercise date: {b.epe_bp[1]:.2f} ± {b.se_bp[1]:.2f} bp     price today: {b.epe_bp[0]:.2f} bp")
print(f"ENE on the first exercise date: {b.ene_bp[1]:.2f} bp (an unexercised option is never a liability)")

Points to be able to make:

- Exposure starts at the option's price, well above the swap's exposure at the same dates: the option is an asset on
  every path, the swap on about half.
- As paths exercise, option exposure is replaced by swap exposure. At the last date the two profiles meet.
- Negative exposure appears only after exercise and is much smaller than the swap's own: the holder chose when to
  enter. That is why the Bermudan's DVA is a fraction of the swap's.
- Exercise frequencies in the table are under the model's pricing measure, not real-world probabilities.

## 2. Sensitivities

**Credit (CS01).** Bump each CDS quote by 1 bp, re-bootstrap the hazard curve, recompute CVA. Exposure does not
change, so this is cheap.

In [ ]:
cs = t["xva_cs01"].set_index("quote")
cs.plot.bar(figsize=(7, 3.2), rot=0); plt.ylabel("CVA change, bp per bp of spread"); plt.axhline(0, color="k", lw=0.8); plt.show()
print("parallel CS01:", cs.sum().round(3).to_dict())

The short-maturity entries are **negative**. Raising the 1-year spread alone, with the others held, moves default
probability from the later years, where the exposure is, into the first year, where there is little. A CVA hedge
built from one CDS maturity would get this wrong.

**Rates.** The delta of a swap's CVA to each of the 37 quotes the curves are built from, three ways:

- *adjoint*: the swaption weights the strip was priced with are already its derivatives to each bond, so CVA is
  linear in the discount factors with known coefficients: one tape sweep and one transposed solve;
- *bump, model fixed*: 37 curve rebuilds, the model's variance held;
- *bump, re-calibrated*: the same, re-fitting the model to the same normal vols each time.

In [ ]:
d, tm = t["xva_delta"], t["xva_delta_timing"].set_index("method").ms
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
big = d[d.adjoint.abs() > 0.005]
x = np.arange(len(big)); a1.bar(x - 0.27, big.adjoint, 0.27, label="adjoint"); a1.bar(x, big.bump_model_fixed, 0.27, label="bump, model fixed"); a1.bar(x + 0.27, big.bump_recalibrated, 0.27, label="bump, re-calibrated")
a1.set_xticks(x, big.quote, rotation=60, ha="right", fontsize=7); a1.set_ylabel("CVA delta, bp per bp"); a1.legend()
a2.barh(tm.index, tm.values); a2.set_xscale("log"); a2.set_xlabel("milliseconds (log scale)"); plt.tight_layout(); plt.show()
print(f"total delta: adjoint {d.adjoint.sum():+.4f}, bump model fixed {d.bump_model_fixed.sum():+.4f}, bump re-calibrated {d.bump_recalibrated.sum():+.4f}")
print(f"bumping is {tm['bump (model fixed)'] / tm['adjoint']:.0f} times slower than the adjoint")

Two things to notice:

- The adjoint and the model-fixed bump agree up to the convexity of a one-sided 1 bp bump (the tests match the
  adjoint to a small central bump to 2e-10).
- Re-calibrating gives a different ladder. Neither is wrong: holding the model fixed is the delta of a desk that
  hedges vega separately; re-calibrating folds part of the vega into the delta.

## 3. Wrong-way risk

Everything so far multiplies expected exposure by default probability, which assumes they are independent. Here the
hazard rate on each interval depends on the rates state $z \sim N(0,1)$ at the end of it:

$$\lambda_j = a_j\, e^{\beta z_j - \beta^2/2},$$

so a one-standard-deviation rise in rates multiplies the hazard by $e^\beta$. The levels $a_j$ are re-solved so that
the CDS curve is still repriced; only the dependence changes.

In [ ]:
w, pr = t["xva_wwr"], t["xva_wwr_profile"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
a1.plot(w.beta, w.payer_cva_bp, "o-", label="10Y payer"); a1.plot(w.beta, w.receiver_cva_bp, "o-", label="10Y receiver"); a1.plot(w.beta, w.book_cva_bp, "o-", label="eight-swap book")
a1.set_xlabel("beta"); a1.set_ylabel("CVA, bp of notional"); a1.legend()
a2.plot(pr.time, pr.epe_given_default_wrong_way_bp, label="given default, beta = +1"); a2.plot(pr.time, pr.epe_bp, "k--", label="unconditional"); a2.plot(pr.time, pr.epe_given_default_right_way_bp, label="given default, beta = −1")
a2.set_title("payer's expected exposure"); a2.set_xlabel("years"); a2.legend(); plt.show()
base = w[w.beta == 0].iloc[0]
w.assign(payer_ratio=w.payer_cva_bp / base.payer_cva_bp, receiver_ratio=w.receiver_cva_bp / base.receiver_cva_bp)[["beta", "hazard_multiple_per_sd", "payer_cva_bp", "payer_ratio", "receiver_cva_bp", "receiver_ratio", "book_cva_bp", "worst_survival_error"]].round(3)

Points to be able to make:

- The payer is owed money when rates are high. If that is also when this counterparty tends to default ($\beta > 0$),
  the exposure that matters, the one **conditional on default**, is well above the average exposure. That is
  wrong-way risk. The receiver faces the same counterparty with right-way risk, and its CVA falls.
- A single multiplier on CVA cannot represent this: the effect has a sign that depends on which way the trade faces.
- `worst_survival_error` is zero to machine precision in every row: the CDS curve is repriced whatever $\beta$ is, so
  the amount of default risk is unchanged and only its timing relative to exposure moves.
- $\beta$ is an input, not an estimate, and one factor drives both rates and credit. The section measures how much
  CVA moves for a stated dependence; choosing the dependence is the hard part.
- The independent CVA here ($\beta = 0$) is higher than in notebook 09's single-swap table because the exposure dates
  are quarterly rather than coupon dates only. Exposure drops each time a coupon rolls off, so a coupon-date grid
  samples the profile at its lowest points.